# Simple calling data analysis 

In [ ]:
import numpy as np
import sklearn
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import os
from PIL import Image
import imagehash
import imageio.v3 as imageio
import pandas as pd
from collections import Counter
import random

Image_size = 96
Batch = 32
Random_seed = 42

In [ ]:
def getImage(hasFire, whichImg):
    """Get one of the training images"""
    
    if hasFire:
        label = "Fire"
        filename = f'Training/Fire/resized_frame{whichImg}.jpg'
    else:
        label = "No Fire"
        filename = f'Training/No_Fire/resized_frame{whichImg}.jpg'
        
        if not os.path.isfile(filename):
            label = "No Fire Lake"
            filename = f'Training/No_Fire/lake_resized_lake_frame{whichImg}.jpg'
            
    return label, imageio.imread(filename)

# Set Figure size and axes
ig, axes = plt.subplots(3, 4, figsize=(16, 12)) 
axes = axes.flatten()


for i in range(12):    
    # Picking Fire then No fire 
    is_fire = i % 2 == 0 
    label, im = getImage(is_fire, i)
    
    # Plotting
    axes[i].imshow(im)
    axes[i].set_title(f"{label} (Index: {i})")
    axes[i].axis('off')  

plt.tight_layout()
plt.show()

# first image size i og used 256 and 128 but take to long :

# Can see as frames alot of repeated data, will compare  picking 1 in every 30 frames to image hashing for model accuracy. Will then compare accuracy of small model with these 2 methods and not removinig any images at all to see a differnce


In [ ]:
def get_clean_data_ImageHash(dirs):
    all_paths, all_labels = [], []
    for d in dirs:

        # Finding folder paths 
        class_names = sorted(os.listdir(d))
        class_to_idx = {name: i for i, name in enumerate(class_names)}
        for class_name in class_names:
            c_path = os.path.join(d, class_name)
            if not os.path.isdir(c_path): continue
            files = [os.path.join(c_path, f) for f in os.listdir(c_path)]
            all_paths.extend(files)
            all_labels.extend([class_to_idx[class_name]] * len(files))
    
    # Use hashing to remove duplicates in all folders
    unique_hashes = {}
    clean_paths, clean_labels = [], []
    print(f"Total raw files: {len(all_paths)}")
    for p, l in zip(all_paths, all_labels):
        try:
            with Image.open(p) as img:
                h = str(imagehash.phash(img))
                if h not in unique_hashes:
                    unique_hashes[h] = p
                    clean_paths.append(p)
                    clean_labels.append(l)
        except: continue
    print(f"Total unique files: {len(clean_paths)}")
    return clean_paths, clean_labels


def get_clean_data_OneIn30(dirs, skip_rate=30):
    all_paths, all_labels = [], []
    
    for d in dirs:
        class_names = sorted(os.listdir(d))
        # Map folder names to indices 
        class_to_idx = {name: i for i, name in enumerate(class_names)}
        
        for class_name in class_names:
            c_path = os.path.join(d, class_name)
            if not os.path.isdir(c_path): continue
            
            # Get files in this specific class folder
            files = sorted([os.path.join(c_path, f) for f in os.listdir(c_path)])
            
            # Pick every 30th image 
            sampled_files = files[::skip_rate] 
            
            all_paths.extend(sampled_files)
            all_labels.extend([class_to_idx[class_name]] * len(sampled_files))
            
    print(f"Total sampled files (1 in {skip_rate}): {len(all_paths)}")
    return all_paths, all_labels

# Analysis size of data 

In [ ]:

def balance_split(paths, labels):
    counts = Counter(labels)
    min_count = min(counts.values())
    combined = list(zip(paths, labels))
    random.shuffle(combined)
    bal_p, bal_l = [], []
    class_seen = Counter()
    for p, l in combined:
        if class_seen[l] < min_count:
            bal_p.append(p)
            bal_l.append(l)
            class_seen[l] += 1
    return bal_p, bal_l

def augment(image, label):
    # Various augmentations, from rotation to random saturation and blurring
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform((), 0, 4, dtype=tf.int32))
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    image = tf.image.random_hue(image, max_delta=0.05)
    
    # Haze - fixed with tf.cond
    haze_intensity = tf.random.uniform((), 0.05, 0.25)
    haze = tf.ones_like(image) * 0.7
    image = tf.cond(
        tf.random.uniform(()) > 0.5,
        lambda: image * (1 - haze_intensity) + haze * haze_intensity,
        lambda: image
    )

    # Blur - fixed with tf.cond
    def apply_blur(img):
        img = tf.expand_dims(img, 0)
        img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
        return tf.squeeze(img, 0)

    image = tf.cond(
        tf.random.uniform(()) > 0.6,
        lambda: apply_blur(image),
        lambda: image
    )

    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label


def normalize(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [Image_size, Image_size])
    return img, label

def prepare_dataset(paths, labels, use_aug=False, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths))
    ds = ds.map(load_and_preprocess)
    if use_aug:
        ds = ds.map(augment)
    ds = ds.map(normalize).batch(Batch).prefetch(tf.data.AUTOTUNE)
    return ds

# first i only tried wieghts and augmentation this made the data ovefit on useless things

In [ ]:
from sklearn.metrics import f1_score


train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    "Training/",
    validation_split=0.15,
    subset="training",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),   # load at 224 directly
    batch_size=None,
    class_names=["No_Fire", "Fire"]
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    "Training/",
    validation_split=0.15,
    subset="validation",
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None,
    class_names=["No_Fire", "Fire"]
)

test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    "Test/",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=None,
    shuffle=False,
    class_names=["No_Fire", "Fire"]
)

def compute_class_weights(ds):
    labels = np.array([y.numpy() for _, y in ds])
    counts = np.bincount(labels)
    total = len(labels)
    weights = {i: total / (len(counts) * c) for i, c in enumerate(counts)}
    print(f"Class weights: {weights}")
    return weights

class_weights = compute_class_weights(train_ds_raw)

# ── Build Final Datasets ──────────────────────────────────────────────────────
train_ds = (
    train_ds_raw
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    val_ds_raw
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
test_ds = (
    test_ds_raw
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

model = keras.models.Sequential([
        keras.layers.Conv2D(8, 3, activation='relu', padding='same', input_shape=(Image_size, Image_size, 3)),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(8, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Recall(name='recall'), keras.metrics.Precision(name='precision')]
    )
    
model.fit(train_ds, epochs=10, validation_data=val_ds, verbose=0)
    
loss, acc, recall, precision, auc = model.evaluate(test_ds)
print(f"Test Loss: {loss:.4f} | Accuracy: {acc:.4f} | Recall: {recall:.4f} | Precision: {precision:.4f} | AUC: {auc:.4f}")

# Get predictions
preds = model.predict(test_ds)
true_labels = np.concatenate([y.numpy() for _, y in test_ds])

# Tune threshold on validation set (not test set)
val_preds = model.predict(val_ds)
val_true = np.concatenate([y.numpy() for _, y in val_ds])

thresholds = np.arange(0.1, 0.9, 0.05)
f1_scores = [f1_score(val_true, (val_preds > t).astype(int)) for t in thresholds]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best threshold: {best_threshold:.2f}  (Val F1={max(f1_scores):.4f})")

plt.plot(thresholds, f1_scores)
plt.axvline(best_threshold, color='r', linestyle='--', label=f'Best = {best_threshold:.2f}')
plt.xlabel('Threshold'); plt.ylabel('F1'); plt.title('Threshold Tuning'); plt.legend()
plt.show()

# Final evaluation using tuned threshold
pred_classes = (preds > best_threshold).astype(int).flatten()
print(confusion_matrix(true_labels, pred_classes))
print(classification_report(true_labels, pred_classes, target_names=['No_Fire', 'Fire']))

In [ ]:
def run_experiment(method='hashing', balancing='weights', use_aug=False):
    print(f"Running: Method={method}, Balancing={balancing}, Augmentation={use_aug}")
    

    if method == 'hashing':
        paths, labels = get_clean_data_ImageHash(["Training/", "Test/"])
    else:
        paths, labels = get_clean_data_OneIn30(["Training/", "Test/"], skip_rate=30)
    

    tr_p, temp_p, tr_l, temp_l = train_test_split(paths, labels, test_size=0.25, random_state=Random_seed, stratify=labels)
    val_p, te_p, val_l, te_l = train_test_split(temp_p, temp_l, test_size=0.5, random_state=Random_seed, stratify=temp_l)
    
    weights = None
    if balancing == 'downsample':
        tr_p, tr_l = balance_split(tr_p, tr_l)
    else:
        cls_weights = compute_class_weight('balanced', classes=np.unique(tr_l), y=tr_l)
        weights = dict(enumerate(cls_weights))
    
    train_ds = prepare_dataset(tr_p, tr_l, use_aug=use_aug)
    val_ds = prepare_dataset(val_p, val_l, use_aug=False)
    test_ds = prepare_dataset(te_p, te_l, use_aug=False, shuffle=False)
    
    model = keras.models.Sequential([
        keras.layers.Conv2D(8, 3, activation='relu', padding='same', input_shape=(Image_size, Image_size, 3)),
        keras.layers.MaxPooling2D(),
        keras.layers.Dropout(0.2),
        keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
        keras.layers.GlobalAveragePooling2D(),
        keras.layers.Dense(8, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.Recall(name='recall'), keras.metrics.Precision(name='precision')]
    )
    
    model.fit(train_ds, epochs=10, validation_data=val_ds, class_weight=weights, verbose=0)
    
    eval_res = model.evaluate(test_ds, verbose=0)
    return {
        'method': method, 'balancing': balancing, 'aug': use_aug,
        'accuracy': eval_res[1], 'recall': eval_res[2], 'precision': eval_res[3]
    }

In [ ]:
experiments = [
    # Hashing Methods
    {'method': 'hashing', 'balancing': 'weights',   'aug': False},
    {'method': 'hashing', 'balancing': 'weights',   'aug': True},
    {'method': 'hashing', 'balancing': 'downsample', 'aug': False},
    {'method': 'hashing', 'balancing': 'downsample', 'aug': True},
    
    # Picking Methods (1 in 30)
    {'method': 'picking', 'balancing': 'weights',   'aug': False},
    {'method': 'picking', 'balancing': 'weights',   'aug': True},
    {'method': 'picking', 'balancing': 'downsample', 'aug': False},
    {'method': 'picking', 'balancing': 'downsample', 'aug': True}
]

results = []
for exp in experiments:
    try:
        res = run_experiment(method=exp['method'], balancing=exp['balancing'], use_aug=exp['aug'])
        results.append(res)
    except Exception as e:
        print(f"Experiment failed for {exp}: {e}")

In [ ]:
df = pd.DataFrame(results)
df['label'] = df.apply(lambda r: f"{r.method}\n{r.balancing}\nAug:{r.aug}", axis=1)

ax = df.plot(x='label', y=['accuracy', 'recall', 'precision'], kind='bar', figsize=(12, 6))
plt.title("Model Performance Comparison Across Data Processing Methods")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# Previously not working ones 

### Did not iniclude downsampling 

### only had weights at start

### varied hue 

### intially no data naylsis but took to long model was bad 


# hue vs no hue as fire is colour sensitive

In [ ]:
def augment_flexible(image, label, use_hue=True):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform((), 0, 4, dtype=tf.int32))
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.4)
    image = tf.image.random_saturation(image, lower=0.6, upper=1.6)
    
    # Toggleable Hue Augmentation
    if use_hue:
        image = tf.image.random_hue(image, max_delta=0.05)
    
    # Haze and Blur
    haze_intensity = tf.random.uniform((), 0.05, 0.25)
    haze = tf.ones_like(image) * 0.7
    image = tf.cond(tf.random.uniform(()) > 0.5, 
                    lambda: image * (1 - haze_intensity) + haze * haze_intensity, 
                    lambda: image)
    
    def apply_blur(img):
        img = tf.expand_dims(img, 0)
        img = tf.nn.avg_pool2d(img, ksize=3, strides=1, padding='SAME')
        return tf.squeeze(img, 0)

    image = tf.cond(tf.random.uniform(()) > 0.6, lambda: apply_blur(image), lambda: image)
    return tf.clip_by_value(image, 0.0, 1.0), label

def run_hue_comparison():
    # Using Hashing + Weights as the baseline from your previous experiments
    paths, labels = get_clean_data_ImageHash(["Training/", "Test/"])
    tr_p, temp_p, tr_l, temp_l = train_test_split(paths, labels, test_size=0.25, random_state=Random_seed, stratify=labels)
    val_p, te_p, val_l, te_l = train_test_split(temp_p, temp_l, test_size=0.5, random_state=Random_seed, stratify=temp_l)
    
    cls_weights = compute_class_weight('balanced', classes=np.unique(tr_l), y=tr_l)
    weights = dict(enumerate(cls_weights))

    results = []
    for use_hue in [True, False]:
        print(f"Training Small Model - Random Hue: {use_hue}")
        
        # Prepare datasets using the flexible augmentation
        train_ds = (tf.data.Dataset.from_tensor_slices((tr_p, tr_l))
                    .shuffle(len(tr_p))
                    .map(load_and_preprocess)
                    .map(lambda x, y: augment_flexible(x, y, use_hue=use_hue))
                    .batch(Batch).prefetch(tf.data.AUTOTUNE))
        
        val_ds = prepare_dataset(val_p, val_l, use_aug=False)
        test_ds = prepare_dataset(te_p, te_l, use_aug=False, shuffle=False)

        # Your 'Small' Model Architecture
        model = keras.models.Sequential([
            keras.layers.Conv2D(8, 3, activation='relu', padding='same', input_shape=(Image_size, Image_size, 3)),
            keras.layers.MaxPooling2D(),
            keras.layers.Dropout(0.2),
            keras.layers.Conv2D(16, 3, activation='relu', padding='same'),
            keras.layers.GlobalAveragePooling2D(),
            keras.layers.Dense(8, activation='relu'),
            keras.layers.Dropout(0.2),
            keras.layers.Dense(1, activation='sigmoid')
        ])

        model.compile(optimizer=keras.optimizers.Adam(1e-4), loss='binary_crossentropy', metrics=['accuracy', 'recall'])
        model.fit(train_ds, epochs=10, validation_data=val_ds, class_weight=weights, verbose=0)
        
        eval_res = model.evaluate(test_ds, verbose=0)
        results.append({'Hue Augmentation': use_hue, 'Accuracy': eval_res[1], 'Recall': eval_res[2]})

    # Display Results
    res_df = pd.DataFrame(results)
    print("\n--- Hue Comparison Results ---")
    print(res_df)
    res_df.plot(kind='bar', x='Hue Augmentation', title='Impact of Hue Augmentation on Fire Detection')
    plt.show()

run_hue_comparison()